# Lab 4: CNN vs MLP na CIFAR-10 (kod)
Ten notebook zawiera **tylko kod i eksperymenty**. Sprawozdanie jest w osobnym pliku `REPORT.md`; notebook zapisuje wyniki do `figures/` i `results/`.

**Uruchom na GPU:** Runtime → Change runtime type → GPU, potem Runtime → Run all (ok. 10–15 minut).
Zbiór testowy jest użyty **tylko raz**, w komórce „Końcowa ocena", po wyborze checkpointów na walidacji.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

SEED = 0
EPOCHS = 15  # ten sam budżet epok dla wszystkich modeli
BATCH = 128
LR = 1e-3  # Adam, ten sam dla wszystkich modeli
device = "cuda" if torch.cuda.is_available() else "cpu"
hardware = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"

FIG_DIR, RES_DIR = Path("figures"), Path("results")
for d in (FIG_DIR, RES_DIR):
    d.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "| sprzęt:", hardware)
if device != "cuda":
    print("UWAGA: brak GPU, trening CNN będzie bardzo wolny!")


def sync():
    # poprawny pomiar czasu na GPU wymaga synchronizacji
    if device == "cuda":
        torch.cuda.synchronize()


def warmup_gpu():
    # rozgrzewka bibliotek CUDA, żeby pierwszy model nie płacił za start
    if device != "cuda":
        return
    m = nn.Sequential(
        nn.Conv2d(3, 8, 3),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(8, 10),
    ).to(device)
    m(torch.randn(64, 3, 32, 32, device=device)).sum().backward()
    sync()


warmup_gpu()

## 1. Dane: CIFAR-10 i stały podział 45 000 / 5 000 / 10 000
Zbiór treningowy (50 000) dzielimy **raz** na 45 000 / 5 000 (walidacja) stałą permutacją. Zbiór testowy (10 000) nie bierze udziału w żadnych decyzjach. Wszystkie dane trzymamy w pamięci GPU, więc czas uczenia mierzy obliczenia, a nie ładowanie danych (przetwarzanie jest identyczne dla obu modeli: tylko normalizacja, bez augmentacji).

In [ ]:
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


def load_split(train):
    ds = torchvision.datasets.CIFAR10("./data", train=train, download=True)
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float().contiguous()
    x = (x / 255.0 - mean) / std
    return x, torch.tensor(ds.targets), ds.classes


x_all, y_all, classes = load_split(True)
x_te, y_te, _ = load_split(False)

split_gen = torch.Generator().manual_seed(42)
perm = torch.randperm(len(x_all), generator=split_gen)
train_idx, val_idx = perm[:45_000], perm[45_000:]
assert set(train_idx.tolist()).isdisjoint(val_idx.tolist())
np.savez(
    RES_DIR / "split_idx.npz", train=train_idx.numpy(), val=val_idx.numpy()
)

x_train, y_train = x_all[train_idx].to(device), y_all[train_idx].to(device)
x_val, y_val = x_all[val_idx].to(device), y_all[val_idx].to(device)
x_test, y_test = x_te.to(device), y_te.to(device)
print("Klasy:", classes)
print(
    f"Trening: {len(x_train)} | walidacja: {len(x_val)} | test: {len(x_test)}"
)
print("Kształt wejścia:", tuple(x_train.shape[1:]))
print("Klasy w walidacji:", torch.bincount(y_val.cpu()).tolist())

## 2. Modele: MLP i CNN
Architektury z ćwiczenia. CNN jest sparametryzowana listą kanałów w blokach (`channels`), domyślnie `(32, 64)`, co daje dokładnie model z ćwiczenia; potrzebujemy tego do eksperymentów dodatkowych.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=3 * 32 * 32, hidden=(512, 256), n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_classes),
        )

    def forward(self, x):
        return self.net(x)


class SimpleCNN(nn.Module):
    def __init__(self, channels=(32, 64), n_classes=10, p_drop=0.3):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )

        blocks, cin = [], 3
        for cout in channels:
            blocks.append(block(cin, cout))
            cin = cout
        self.features = nn.Sequential(*blocks)
        side = 32 // 2 ** len(channels)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(cin * side * side, 256),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def count_params(m):
    return sum(p.numel() for p in m.parameters())


print("Parametry MLP:", f"{count_params(MLP()):,}")
print("Parametry CNN:", f"{count_params(SimpleCNN()):,}")

## 3. Trening z pomiarem czasu i wyborem checkpointu na walidacji
Jedna funkcja `run` dla wszystkich modeli. Po każdej epoce liczymy stratę i accuracy na walidacji; zapamiętujemy checkpoint z **najniższą stratą walidacyjną**. Czas mierzymy tylko dla pętli treningowej (z synchronizacją GPU), bez ewaluacji.

In [ ]:
def batches(x, y, batch_size, gen=None):
    n = len(x)
    if gen is None:
        idx = torch.arange(n, device=x.device)
    else:
        idx = torch.randperm(n, generator=gen).to(x.device)
    for i in range(0, n, batch_size):
        j = idx[i : i + batch_size]
        yield x[j], y[j]


@torch.no_grad()
def evaluate(model, x, y, batch_size=1000):
    model.eval()
    crit = nn.CrossEntropyLoss(reduction="sum")
    loss_sum, correct = 0.0, 0
    for xb, yb in batches(x, y, batch_size):
        out = model(xb)
        loss_sum += crit(out, yb).item()
        correct += (out.argmax(1) == yb).sum().item()
    return loss_sum / len(x), correct / len(x)


def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def run(name, factory, epochs=EPOCHS, lr=LR, seed=SEED, verbose=True):
    torch.manual_seed(seed)  # ta sama inicjalizacja i kolejność batchy
    model = factory().to(device)
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    keys = ["train_loss", "train_acc", "val_loss", "val_acc", "epoch_time"]
    hist = {k: [] for k in keys}
    best_val, best_epoch, best_state = float("inf"), 0, None

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = torch.zeros((), device=device)
        correct = torch.zeros((), device=device)
        sync()
        t0 = time.perf_counter()
        for xb, yb in batches(x_train, y_train, BATCH, gen):
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            loss_sum += loss.detach() * yb.numel()
            correct += (out.argmax(1) == yb).sum()
        sync()
        hist["epoch_time"].append(time.perf_counter() - t0)
        hist["train_loss"].append(loss_sum.item() / len(x_train))
        hist["train_acc"].append(correct.item() / len(x_train))

        val_loss, val_acc = evaluate(model, x_val, y_val)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)
        if val_loss < best_val:  # wybór epoki po stracie walidacyjnej
            best_val, best_epoch = val_loss, epoch
            best_state = clone_state(model)
        if verbose:
            print(
                f"  epoka {epoch:2d} | trening {hist['train_loss'][-1]:.3f} | "
                f"val {val_loss:.3f} | val acc {val_acc * 100:.2f}% | "
                f"{hist['epoch_time'][-1]:.1f} s"
            )

    return {
        "name": name,
        "factory": factory,
        "params": count_params(model),
        "hist": hist,
        "best_epoch": best_epoch,
        "best_state": best_state,
        "last_state": clone_state(model),
        "train_time": sum(hist["epoch_time"]),
    }


def summarize(res):
    h, i = res["hist"], res["best_epoch"] - 1
    return {
        "model": res["name"],
        "parametry": res["params"],
        "wybrana epoka": res["best_epoch"],
        "val acc [%]": round(h["val_acc"][i] * 100, 2),
        "val loss": round(h["val_loss"][i], 3),
        "czas uczenia [s]": round(res["train_time"], 1),
        "czas / epokę [s]": round(res["train_time"] / len(h["epoch_time"]), 2),
    }

## 4. Trening: MLP vs CNN
Ten sam optymalizator (Adam, lr = 1e-3), batch 128, **15 epok**, ten sam seed i te same dane.

In [ ]:
results = {}
print("== MLP ==")
results["MLP"] = run("MLP", MLP)
print("\n== CNN ==")
results["CNN"] = run("CNN", SimpleCNN)

## 5. Krzywe uczenia: MLP vs CNN (walidacja)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
colors = {"MLP": "tab:orange", "CNN": "tab:blue"}
for name in ("MLP", "CNN"):
    h = results[name]["hist"]
    ep = range(1, len(h["val_acc"]) + 1)
    c = colors[name]
    ax1.plot(
        ep,
        [a * 100 for a in h["val_acc"]],
        "-o",
        color=c,
        ms=3,
        label=f"{name} walidacja",
    )
    ax1.plot(
        ep,
        [a * 100 for a in h["train_acc"]],
        "--",
        color=c,
        label=f"{name} trening",
    )
    ax1.axvline(results[name]["best_epoch"], color=c, ls=":", lw=0.8)
    ax2.plot(ep, h["val_loss"], "-o", color=c, ms=3, label=f"{name} walidacja")
    ax2.plot(ep, h["train_loss"], "--", color=c, label=f"{name} trening")
ax1.axhline(10, color="gray", lw=0.8)
ax1.text(1, 11, "losowy baseline (10%)", fontsize=8, color="gray")
ax1.set_xlabel("epoka")
ax1.set_ylabel("accuracy [%]")
ax1.set_title("Accuracy (linie pionowe: wybrana epoka)")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)
ax2.set_xlabel("epoka")
ax2.set_ylabel("strata (CE)")
ax2.set_title("Strata")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "curves_mlp_vs_cnn.png", dpi=150)
plt.show()

## 6. Końcowa ocena na zbiorze testowym (decyzja zamrożona)
**Uwaga:** to jedyne miejsce, w którym używamy zbioru testowego. Oceniamy checkpointy wybrane na walidacji (najniższa strata walidacyjna). Dla przejrzystości podajemy też accuracy testową po ostatniej epoce, ale nie służy ona do wyboru modelu.

In [ ]:
rows = []
for name in ("MLP", "CNN"):
    r = results[name]
    model = r["factory"]().to(device)
    model.load_state_dict(r["best_state"])
    _, acc_ckpt = evaluate(model, x_test, y_test)
    model.load_state_dict(r["last_state"])
    _, acc_last = evaluate(model, x_test, y_test)
    s = summarize(r)
    rows.append(
        {
            "model": name,
            "parametry": s["parametry"],
            "wybrana epoka": s["wybrana epoka"],
            "val acc [%]": s["val acc [%]"],
            "test acc [%] (checkpoint)": round(acc_ckpt * 100, 2),
            "test acc [%] (ostatnia epoka)": round(acc_last * 100, 2),
            "nad losowym [p.p.]": round(acc_ckpt * 100 - 10, 2),
            "czas uczenia [s]": s["czas uczenia [s]"],
            "czas / epokę [s]": s["czas / epokę [s]"],
        }
    )
final_df = pd.DataFrame(rows)
final_df.to_csv(RES_DIR / "main_comparison.csv", index=False)
display(final_df)
print(final_df.to_markdown(index=False))

config = {
    "zbior": "CIFAR-10",
    "podzial": "45000 / 5000 / 10000 (seed 42)",
    "optymalizator": "Adam",
    "lr": LR,
    "batch": BATCH,
    "epoki": EPOCHS,
    "seed": SEED,
    "wybor_checkpointu": "najnizsza strata walidacyjna",
    "sprzet": hardware,
    "pytorch": torch.__version__,
}
json.dump(config, open(RES_DIR / "run_config.json", "w"), indent=2)
print(json.dumps(config, indent=2, ensure_ascii=False))

## 7. Zakres dodatkowy: liczba filtrów i liczba bloków
Wszystkie warianty: ten sam budżet (15 epok), optymalizator i dane; oceniamy na **walidacji** (zbiór testowy nie jest używany). Liczbę filtrów zmieniamy parami (pierwszy blok, drugi blok) jako 16–32, 32–64 (baza) i 64–128; liczbę bloków jako 1, 2 (baza) i 3.

In [ ]:
extras = {
    "filtry 16-32": lambda: SimpleCNN((16, 32)),
    "filtry 64-128": lambda: SimpleCNN((64, 128)),
    "1 blok (32)": lambda: SimpleCNN((32,)),
    "3 bloki (32-64-128)": lambda: SimpleCNN((32, 64, 128)),
}
for name, factory in extras.items():
    print("==", name, "==")
    results[name] = run(name, factory, verbose=False)
    s = summarize(results[name])
    print(
        f"  parametry {s['parametry']:,} | val acc {s['val acc [%]']}% | "
        f"epoka {s['wybrana epoka']} | czas {s['czas uczenia [s]']} s"
    )

In [ ]:
order = [
    ("filtry 16-32", "filtry"),
    ("CNN", "filtry"),
    ("filtry 64-128", "filtry"),
    ("1 blok (32)", "bloki"),
    ("CNN", "bloki"),
    ("3 bloki (32-64-128)", "bloki"),
]
rows = []
for name, group in order:
    s = summarize(results[name])
    s["eksperyment"] = group
    if name == "CNN":
        s["model"] = "CNN (baza: filtry 32-64, 2 bloki)"
    rows.append(s)
ex_df = pd.DataFrame(rows)[
    [
        "eksperyment",
        "model",
        "parametry",
        "wybrana epoka",
        "val acc [%]",
        "val loss",
        "czas uczenia [s]",
    ]
]
ex_df.to_csv(RES_DIR / "extras_filters_blocks.csv", index=False)
display(ex_df)
print(ex_df.to_markdown(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, group in zip(axes, ("filtry", "bloki")):
    sub = ex_df[ex_df["eksperyment"] == group]
    ax.plot(sub["parametry"], sub["val acc [%]"], "o-")
    for _, r in sub.iterrows():
        ax.annotate(
            r["model"].split(" (baza")[0],
            (r["parametry"], r["val acc [%]"]),
            fontsize=8,
            xytext=(4, 4),
            textcoords="offset points",
        )
    ax.set_xlabel("liczba parametrów")
    ax.set_ylabel("val accuracy [%]")
    ax.set_title(f"Wpływ liczby: {group}")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "extras_filters_blocks.png", dpi=150)
plt.show()

## 8. Zakres dodatkowy: mapy cech pierwszej i głębszej warstwy
Przepuszczamy jeden obraz **walidacyjny** przez CNN z wybranym checkpointem i pokazujemy 8 map z pierwszej warstwy splotowej (po BN i ReLU, 32×32) oraz 8 z głębszej (druga warstwa splotowa drugiego bloku, 16×16).

In [ ]:
cnn = SimpleCNN().to(device)
cnn.load_state_dict(results["CNN"]["best_state"])
cnn.eval()

acts = {}


def make_hook(key):
    def hook(module, inputs, output):
        acts[key] = output.detach().cpu()

    return hook


h1 = cnn.features[0][2].register_forward_hook(make_hook("pierwsza"))
h2 = cnn.features[1][5].register_forward_hook(make_hook("glebsza"))
img, label = x_val[:1], y_val[0].item()
with torch.no_grad():
    pred = cnn(img).argmax(1).item()
h1.remove()
h2.remove()

rgb = (img.cpu() * std + mean)[0].clamp(0, 1).permute(1, 2, 0).numpy()
fig, axes = plt.subplots(3, 9, figsize=(14, 5.4))
for ax in axes.flat:
    ax.axis("off")
axes[0, 0].imshow(rgb)
axes[0, 0].set_title(
    f"wejście: {classes[label]}\npred: {classes[pred]}", fontsize=8
)
for k in range(8):
    axes[1, k + 1].imshow(acts["pierwsza"][0, k], cmap="viridis")
    axes[1, k + 1].set_title(f"filtr {k}", fontsize=8)
    axes[2, k + 1].imshow(acts["glebsza"][0, k], cmap="viridis")
    axes[2, k + 1].set_title(f"filtr {k}", fontsize=8)
axes[1, 0].text(0.5, 0.5, "1. warstwa\n(32x32)", ha="center", va="center")
axes[2, 0].text(0.5, 0.5, "głębsza\n(16x16)", ha="center", va="center")
plt.tight_layout()
plt.savefig(FIG_DIR / "feature_maps.png", dpi=150)
plt.show()

## 9. Pliki wynikowe
Do sprawozdania trafiają: `figures/*.png` oraz `results/*.csv|json`. Kod i sprawozdanie są osobnymi plikami.